# NMF on ModulAir 00661

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00661
data = pd.read_csv('/content/MOD-00661.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T09:11:49Z,577187806,2025-12-31T04:11:49Z,MOD-00661,43.1,-1.7,1.536,0.110,0.062,0.017,0.017,...,35.804,17.819,14081.0,14082.0,14083.0,14413.0,14415.0,14419.0,14422.0,2.25
2025-12-31T09:10:49Z,577187809,2025-12-31T04:10:49Z,MOD-00661,43.4,-1.7,1.639,0.119,0.043,0.011,0.009,...,36.028,18.120,14081.0,14082.0,14083.0,14413.0,14415.0,14419.0,14422.0,3.94
2025-12-31T09:09:49Z,577187807,2025-12-31T04:09:49Z,MOD-00661,43.2,-1.7,1.573,0.102,0.029,0.015,0.009,...,36.273,18.493,14081.0,14082.0,14083.0,14413.0,14415.0,14419.0,14422.0,3.86
2025-12-31T09:08:49Z,577187808,2025-12-31T04:08:49Z,MOD-00661,43.3,-1.7,1.719,0.137,0.033,0.012,0.009,...,35.558,18.134,14081.0,14082.0,14083.0,14413.0,14415.0,14419.0,14422.0,3.75
2025-12-31T09:07:49Z,577186019,2025-12-31T04:07:49Z,MOD-00661,43.4,-1.7,1.704,0.127,0.049,0.006,0.012,...,35.554,17.775,14081.0,14082.0,14083.0,14413.0,14415.0,14419.0,14422.0,2.44


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T09:11:49Z,2025-12-31T04:11:49Z,721.168,1.838,35.804,17.819,1.536,0.110,0.062,0.017,0.017,0.000
2025-12-31T09:10:49Z,2025-12-31T04:10:49Z,764.879,1.865,36.028,18.120,1.639,0.119,0.043,0.011,0.009,0.003
2025-12-31T09:09:49Z,2025-12-31T04:09:49Z,741.212,1.864,36.273,18.493,1.573,0.102,0.029,0.015,0.009,0.009
2025-12-31T09:08:49Z,2025-12-31T04:08:49Z,721.847,1.864,35.558,18.134,1.719,0.137,0.033,0.012,0.009,0.006
2025-12-31T09:07:49Z,2025-12-31T04:07:49Z,722.186,1.865,35.554,17.775,1.704,0.127,0.049,0.006,0.012,0.009


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T04:11:49Z,721.168,1.838,35.804,17.819,1.536,0.110,0.062,0.017,0.017,0.000
1,2025-12-31T04:10:49Z,764.879,1.865,36.028,18.120,1.639,0.119,0.043,0.011,0.009,0.003
2,2025-12-31T04:09:49Z,741.212,1.864,36.273,18.493,1.573,0.102,0.029,0.015,0.009,0.009
3,2025-12-31T04:08:49Z,721.847,1.864,35.558,18.134,1.719,0.137,0.033,0.012,0.009,0.006
4,2025-12-31T04:07:49Z,722.186,1.865,35.554,17.775,1.704,0.127,0.049,0.006,0.012,0.009


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 04:11:49,721.168,1.838,35.804,17.819,1.536,0.110,0.062,0.017,0.017,0.000
1,2025-12-31 04:10:49,764.879,1.865,36.028,18.120,1.639,0.119,0.043,0.011,0.009,0.003
2,2025-12-31 04:09:49,741.212,1.864,36.273,18.493,1.573,0.102,0.029,0.015,0.009,0.009
3,2025-12-31 04:08:49,721.847,1.864,35.558,18.134,1.719,0.137,0.033,0.012,0.009,0.006
4,2025-12-31 04:07:49,722.186,1.865,35.554,17.775,1.704,0.127,0.049,0.006,0.012,0.009


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,871.01,34.61,23.72,1.91,20.74,2.40,1.14,0.46,0.64,0.49
1,2025-03-31 21:00:00,983.12,40.10,22.07,2.83,26.92,4.51,1.59,0.55,0.73,0.56
2,2025-03-31 22:00:00,1118.08,40.49,26.15,3.24,25.51,6.41,1.92,0.55,0.65,0.48
3,2025-03-31 23:00:00,852.07,31.95,38.87,2.88,15.63,3.56,1.07,0.28,0.30,0.19
4,2025-04-01 00:00:00,777.47,28.59,42.92,2.83,14.37,2.94,0.84,0.21,0.22,0.14
...,...,...,...,...,...,...,...,...,...,...,...
6422,2025-12-31 00:00:00,735.35,35.48,19.01,1.78,1.34,0.16,0.06,0.01,0.01,0.01
6423,2025-12-31 01:00:00,727.35,35.13,19.07,1.82,1.31,0.14,0.05,0.01,0.01,0.01
6424,2025-12-31 02:00:00,741.47,35.52,18.27,1.86,1.39,0.13,0.05,0.01,0.01,0.00
6425,2025-12-31 03:00:00,735.43,35.37,18.56,1.86,1.55,0.13,0.05,0.01,0.01,0.01


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,871.01,34.61,23.72,1.91,20.74,2.40,1.14,0.46,0.64,0.49
2025-03-31 21:00:00,983.12,40.10,22.07,2.83,26.92,4.51,1.59,0.55,0.73,0.56
2025-03-31 22:00:00,1118.08,40.49,26.15,3.24,25.51,6.41,1.92,0.55,0.65,0.48
2025-03-31 23:00:00,852.07,31.95,38.87,2.88,15.63,3.56,1.07,0.28,0.30,0.19
2025-04-01 00:00:00,777.47,28.59,42.92,2.83,14.37,2.94,0.84,0.21,0.22,0.14


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.415279,0.581095,0.235785,0.014742,0.284851,0.092237,0.094921,0.134111,0.234432,0.345070
2025-03-31 21:00:00,0.468730,0.673271,0.219384,0.021843,0.369729,0.173328,0.132390,0.160350,0.267399,0.394366
2025-03-31 22:00:00,0.533077,0.679819,0.259940,0.025008,0.350364,0.246349,0.159867,0.160350,0.238095,0.338028
2025-03-31 23:00:00,0.406249,0.536434,0.386382,0.022229,0.214668,0.136818,0.089092,0.081633,0.109890,0.133803
2025-04-01 00:00:00,0.370681,0.480020,0.426640,0.021843,0.197363,0.112990,0.069942,0.061224,0.080586,0.098592


In [11]:
data.to_csv('MOD-00661_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.438254,0.571242,0.242854,0.023460,0.214613,0.191229,0.164327,0.166621,0.218110,0.251764
2025-03-31 21:00:00,0.507841,0.656002,0.220450,0.027036,0.300037,0.252231,0.205419,0.202811,0.258797,0.295047
2025-03-31 22:00:00,0.543000,0.675812,0.261757,0.028779,0.330935,0.255692,0.200612,0.194348,0.246207,0.281016
2025-03-31 23:00:00,0.447471,0.516930,0.372198,0.023763,0.204323,0.120056,0.089564,0.084861,0.113072,0.136171
2025-04-01 00:00:00,0.421653,0.455890,0.408883,0.022233,0.185053,0.093491,0.066729,0.061745,0.084756,0.105151
...,...,...,...,...,...,...,...,...,...,...
2025-12-30 22:00:00,0.351788,0.588518,0.201201,0.020425,0.026919,0.000566,0.000230,0.001438,0.009150,0.021122
2025-12-30 23:00:00,0.347472,0.583696,0.197135,0.020194,0.025093,0.000000,0.000000,0.001317,0.008975,0.020828
2025-12-31 00:00:00,0.349539,0.595790,0.187568,0.020380,0.023544,0.000000,0.000000,0.001344,0.008796,0.020554


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.053343,0.027070,0.050140,0.118007
1,0.060173,0.023778,0.074744,0.140215
2,0.061862,0.028881,0.081564,0.131750
3,0.049072,0.043697,0.041815,0.055553
4,0.043512,0.048707,0.034856,0.039150
...,...,...,...,...
6348,0.059431,0.021396,0.000343,0.000000
6349,0.058965,0.020914,0.000000,0.000000
6350,0.060186,0.019623,0.000000,0.000000
6351,0.059550,0.019690,0.000000,0.000000


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,4.837211,9.899097,0.535420,0.294659,0.000000,0.000000,0.000000,0.022336,0.077136,0.208116
Feature 2,2.976340,0.000000,7.916355,0.134820,1.199804,0.000000,0.000000,0.000000,0.211644,0.409128
Feature 3,1.819147,0.594596,0.000000,0.081615,3.632532,1.649136,0.670215,0.323257,0.109542,0.000000
Feature 4,0.071540,0.113403,0.000000,0.000000,0.000000,0.919791,1.107757,1.264519,1.718322,1.945549


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.053343,0.027070,0.050140,0.118007
2025-03-31 21:00:00,0.060173,0.023778,0.074744,0.140215
2025-03-31 22:00:00,0.061862,0.028881,0.081564,0.131750
2025-03-31 23:00:00,0.049072,0.043697,0.041815,0.055553
2025-04-01 00:00:00,0.043512,0.048707,0.034856,0.039150
...,...,...,...,...
2025-12-30 22:00:00,0.059431,0.021396,0.000343,0.000000
2025-12-30 23:00:00,0.058965,0.020914,0.000000,0.000000
2025-12-31 00:00:00,0.060186,0.019623,0.000000,0.000000


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,4.837211,9.899097,0.535420,0.294659,0.000000,0.000000,0.000000,0.022336,0.077136,0.208116
Factor 2,2.976340,0.000000,7.916355,0.134820,1.199804,0.000000,0.000000,0.000000,0.211644,0.409128
Factor 3,1.819147,0.594596,0.000000,0.081615,3.632532,1.649136,0.670215,0.323257,0.109542,0.000000
Factor 4,0.071540,0.113403,0.000000,0.000000,0.000000,0.919791,1.107757,1.264519,1.718322,1.945549


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.575781,0.263101,0.137004,0.003134,0.020980
no,0.656573,0.223098,0.115064,0.000000,0.005265
no2,0.967336,0.000000,0.036763,0.004079,-0.008177
o3,0.084249,0.925065,0.000000,0.000000,-0.009314
bin0,0.000000,0.285488,0.736395,0.000000,-0.021883
bin1,0.000000,0.000000,0.841953,0.273176,-0.115130
bin2,0.000000,0.000000,0.551470,0.530243,-0.081713
bin3,0.032545,0.000000,0.298009,0.678156,-0.008710
bin4,0.082158,0.167409,0.073820,0.673633,0.002980
bin5,0.168843,0.246498,0.000000,0.580956,0.003703


In [20]:
results.to_csv('MOD-00661_factor_results.csv')
comp.to_csv('MOD-00661_factor_comp.csv')
res.to_csv('MOD-00661_factor_resid.csv')